<a href="https://colab.research.google.com/github/juuaa01/NLP/blob/main/NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**TUGAS PRAKTIKUM NLP**

##**Social Media Analytics**

##**Regex untuk Data Twitter/X — Bahasa Gaul, Noise & HTML**

###**NAMA: NAZWAH LAEZA CAMELIA**

###**NIM: 11230940000034**


# BAGIAN A — Tulis Kode Regexnya [Soal 1 – 10]

## Soal 1 [5 poin] — Ekstrak Mention & Hashtag dari Tweet Kotor

In [ ]:
import re
from collections import Counter

tweets = [
 "gak ngerti kenapa @budi_NLP selalu bener prediksinya wkwk #NLP #DeepLearning emg dia sih",
 "RT @sari99: btw tutorial @tensorflow_id kemarin bagus bgt!! #AI #MachineLearning #ML2024",
 "email gw di contact@gmail.com ya, bukan @contact soalnya itu akun lain hehe #random",
 "&lt;p&gt;check @nlp_indo&lt;/p&gt; &amp; jangan lupa #NLP #nlp udh beda hashtag ga??",
 "@@invalid mention ini, tapi @valid_user bisa. ##juga invalid, #validHashtag oke",
]

# Tulis regex buat mention & hashtag di sini
mention_pattern = re.compile(r'(?<![@\w])@[A-Za-z0-9_]+')
hashtag_pattern = re.compile(r'(?<![#\w])#[A-Za-z_][A-Za-z0-9_]*')

all_mentions = []
all_hashtags = []

for tweet in tweets:
  # 1. strip HTML entities dulu
  clean = re.sub(r'&lt;', '<', tweet)
  clean = re.sub(r'&gt;', '>', tweet)
  clean = re.sub(r'&amp', '&', tweet)

  # 2. ekstrak mention & hashtag
  mentions = mention_pattern.findall(clean)
  hashtags = hashtag_pattern.findall(clean)

  all_mentions.extend(mentions)
  all_hashtags.extend(hashtags)


print('Top mentions:', Counter(all_mentions).most_common(5))
print('Top hashtags:', Counter(all_hashtags).most_common(5))

Top mentions: [('@budi_NLP', 1), ('@sari99', 1), ('@tensorflow_id', 1), ('@contact', 1), ('@nlp_indo', 1)]
Top hashtags: [('#NLP', 2), ('#DeepLearning', 1), ('#AI', 1), ('#MachineLearning', 1), ('#ML2024', 1)]


## Soal 2 [5 poin] — Normalisasi Bahasa Alay

In [ ]:
import re

raw_tweets = [
"gak nyangkaaaaa dia bakal m3nang hahahahaha gilaaaa banget sih :((",
"s3rius deh @budi gw 4kan ch3ck d4ta ny4 b3sok pagi oke!!!!!",
"wkwkwkwkwkwk lu emg gila banget sih huhuhuhuhu ngakak poll",
"yaaaaaaaaaa Allah minta di-f0ll0w balik dooongggg #bucin #galau",
"t1dak b1sa g1n1 t3rus, k4pan m4u m4ju cobaaaaa #curcol",
]

def normalize_alay(text: str) -> str:
  # (a) normalisasi elongasi
  text = re.sub(r'(.)\1{2,}', r'\1\1', text)

  # (b) leet speak -> huruf biasa
  leet_map = {
      '4': 'a',
      '3': 'e',
      '1': 'i',
      '0': 'o',
      '5': 's'
  }

  text = re.sub(r'[43105]', lambda x: leet_map[x.group()], text)

  # (c) spasi berulang
  text = re.sub(r'\s+', ' ', text)
  return text

for t in raw_tweets:
  print('BEFORE:', t)
  print('AFTER :', normalize_alay(t))
  print()

BEFORE: gak nyangkaaaaa dia bakal m3nang hahahahaha gilaaaa banget sih :((
AFTER : gak nyangkaa dia bakal menang hahahahaha gilaa banget sih :((

BEFORE: s3rius deh @budi gw 4kan ch3ck d4ta ny4 b3sok pagi oke!!!!!
AFTER : serius deh @budi gw akan check data nya besok pagi oke!!

BEFORE: wkwkwkwkwkwk lu emg gila banget sih huhuhuhuhu ngakak poll
AFTER : wkwkwkwkwkwk lu emg gila banget sih huhuhuhuhu ngakak poll

BEFORE: yaaaaaaaaaa Allah minta di-f0ll0w balik dooongggg #bucin #galau
AFTER : yaa Allah minta di-follow balik doongg #bucin #galau

BEFORE: t1dak b1sa g1n1 t3rus, k4pan m4u m4ju cobaaaaa #curcol
AFTER : tidak bisa gini terus, kapan mau maju cobaa #curcol



## Soal 3 [5 poin] — Strip HTML dari Tweet Crawled

In [ ]:
import re

dirty_tweets = [
    "<b>gak nyangka</b> @budi_ml nge-RT gw!! &lt;3 &amp; thx bgt #blessed",
    "baca dong &lt;a href='https://t.co/abc123'&gt;thread-nya&lt;/a&gt; @sari99 #NLP",
    "mampus&#128514; &#x1F62D; gw ketawa aja liat prediksi @model_gue error 99% &#x1F926;",
    "<span class=\"highlight\">breaking</span>: @openai ngeluarin model baru&nbsp;!!",
    "RT &lt;blockquote&gt;@andi: ini salah&lt;/blockquote&gt; &amp;amp; setuju banget",
]

def clean_tweet(text: str) -> str:

    # 1. simpan URL dulu agar tidak hilang
    urls = re.findall(r'https?://[^\s\'"]+', text)

    # 2. decode HTML entities
    text = re.sub(r'&lt;', '<', text)
    text = re.sub(r'&gt;', '>', text)
    text = re.sub(r'&amp;', '&', text)
    text = re.sub(r'&amp;', '&', text)   # untuk menangani &amp;amp;
    text = re.sub(r'&nbsp;', ' ', text)

    # 3. hapus numeric entity
    text = re.sub(r'&#\d+;', '', text)

    # 4. hapus hex entity
    text = re.sub(r'&#x[0-9A-Fa-f]+;', '', text)

    # 5. hapus tag HTML
    text = re.sub(r'<[^>]+>', '', text)

    # 6. kembalikan URL
    if urls:
        text += ' ' + ' '.join(urls)

    # 7. rapikan spasi
    text = re.sub(r'\s+', ' ', text)

    # 8. hilangkan spasi sebelum tanda baca
    text = re.sub(r'\s+([!?.,:;])', r'\1', text)

    return text.strip()

for d in dirty_tweets:
    print("IN :", d)
    print("OUT:", clean_tweet(d))
    print()

IN : <b>gak nyangka</b> @budi_ml nge-RT gw!! &lt;3 &amp; thx bgt #blessed
OUT: gak nyangka @budi_ml nge-RT gw!! <3 & thx bgt #blessed

IN : baca dong &lt;a href='https://t.co/abc123'&gt;thread-nya&lt;/a&gt; @sari99 #NLP
OUT: baca dong thread-nya @sari99 #NLP https://t.co/abc123

IN : mampus&#128514; &#x1F62D; gw ketawa aja liat prediksi @model_gue error 99% &#x1F926;
OUT: mampus gw ketawa aja liat prediksi @model_gue error 99%

IN : <span class="highlight">breaking</span>: @openai ngeluarin model baru&nbsp;!!
OUT: breaking: @openai ngeluarin model baru!!

IN : RT &lt;blockquote&gt;@andi: ini salah&lt;/blockquote&gt; &amp;amp; setuju banget
OUT: RT @andi: ini salah & setuju banget



## Soal 4 [5 poin] — Ekstrak URL & Klasifikasi

In [ ]:
import re

tweets_url = [
"cek thread lengkapnya di https://t.co/xK9mN2pQrs sama bit.ly/nlp2024 ya guys",
"foto eventnya: https://pbs.twimg.com/media/abc123.jpg <-- keren abis!!",
"paper-nya ada di https://arxiv.org/abs/2310.12345 #NLP #AI wajib baca fr fr",
"&lt;a href=\"https://video.twimg.com/tweet_video/xyz.mp4\"&gt;video&lt;/a&gt;",
"follow juga @nlp_id yg ada di https://nlp-community.id/join?ref=twitter&src=web",
]

URL_PATTERN = re.compile(r'(https?://[^\s"<]+|(?:t\.co|bit\.ly)/[^\s"<]+)')

def classify_url(url: str) -> str:
  # return 'shortener' | 'media' | 'other'
  if 't.co' in url or 'bit.ly' in url:
    return 'shortener'
  elif re.search(r'\.(jpg|jpeg|png|gif|mp4|webp)(\?|$)', url):
    return 'media'
  else:
    return 'other'

for tweet in tweets_url:
  urls = URL_PATTERN.findall(tweet)
  for url in urls:
    print(f' [{classify_url(url)}] {url}')

 [shortener] https://t.co/xK9mN2pQrs
 [shortener] bit.ly/nlp2024
 [media] https://pbs.twimg.com/media/abc123.jpg
 [other] https://arxiv.org/abs/2310.12345
 [media] https://video.twimg.com/tweet_video/xyz.mp4
 [other] https://nlp-community.id/join?ref=twitter&src=web


## Soal 5 [5 poin] — Deteksi Hate Speech Pattern

In [ ]:
import re

suspect_tweets = [
"dasar b*b* lu @rival_user ga tau malu emg!!!!!",
"setuju sama @budi_nlp analisisnya bagus kok, jangan dengerin yg nyinyir",
"@targetuser lo tu a**ing banget sumpah, kapan tobatnya???????????",
"wkwk f**k yg bilang model ini jelek, terbukti kan sekarang hahaha",
"excited banget buat #NLP2024 conference!! semoga bisa hadir :D",
"@user123 emang d*mb*ss lo, ga ngerti-ngerti juga sih padahal udh dijelasin",
]

CENSORED_SLUR = re.compile(r'\b[a-z]{1,3}[*@#$%0-9]{1,4}[a-z]*\b', re.IGNORECASE)
MENTION = re.compile(r'@[A-Za-z_]\w*')
PROVOCATIVE_PUNCT = re.compile(r'[!?]{4,}')

def detect_hate_patterns(tweet: str) -> dict:
  results = {
      'censored_slur': [], # kata yg disensor
      'personal_attack': [], # @user + kasar
      'provocative_punct': False,
      'risk_level': 'low'
      }

  # tulis logika regex di sini
  censored_words = CENSORED_SLUR.findall(tweet)
  results['censored_slur'] = censored_words

  results['provocative_punct'] = bool(PROVOCATIVE_PUNCT.search(tweet))

  # personal attack: ada mention dan ada kata kasar tersensor dalam tweet yang sama
  mentions = MENTION.findall(tweet)
  if mentions and censored_words:
    results['personal_attack'] = mentions

  score = 0
  if results['censored_slur']:
    score += 1
  if results['personal_attack']:
    score += 1
  if results['provocative_punct']:
    score += 1

  if score >= 2:
    results['risk_level'] = 'high'
  elif score == 1:
    results['risk_level'] = 'medium'
  else:
    results['risk_level'] = 'low'

  return results

for tw in suspect_tweets:
  result = detect_hate_patterns(tw)
  print(tw[:60], '->', result['risk_level'])

dasar b*b* lu @rival_user ga tau malu emg!!!!! -> high
setuju sama @budi_nlp analisisnya bagus kok, jangan dengerin -> low
@targetuser lo tu a**ing banget sumpah, kapan tobatnya?????? -> high
wkwk f**k yg bilang model ini jelek, terbukti kan sekarang h -> medium
excited banget buat #NLP2024 conference!! semoga bisa hadir  -> medium
@user123 emang d*mb*ss lo, ga ngerti-ngerti juga sih padahal -> high


## Soal 6 [5 poin] — Parse Tweet Metadata dari Raw HTML Scraping


In [ ]:
import re

html_snippets = [
'<article data-tweet-id="1234567890">'
'<a href="/budi_nlp"><span class="display-name">Budi NLP</span></a>'
'<time datetime="2024-03-15T09:23:41.000Z">15 Mar</time>'
'<div class="tweet-text">gila modelnya overfit parah, loss 0.001 tapi acc test cuma 55% XD #NLP</div>'
'<span class="like-count">1.2K</span>'
'<span class="rt-count">342</span>'
'<span class="reply-count">89</span></article>',

'<article data-tweet-id="9876543210">'
'<a href="/sari_ml99"><span class="display-name">Sari &amp; ML</span></a>'
'<time datetime="2024-03-15T11:05:17.000Z">15 Mar</time>'
'<div class="tweet-text">@budi_nlp regularisasi bro, pake dropout + early stopping &#128514;</div>'
'<span class="like-count">892</span>'
'<span class="rt-count">201</span>'
'<span class="reply-count">47</span></article>',
]

tweet_pattern = re.compile(r'''
    <article\s+data-tweet-id="(?P<tweet_id>\d+)">
    .*?
    <a\s+href="/(?P<username>[A-Za-z0-9_]+)">
    .*?
    <span\s+class="display-name">(?P<display_name>.*?)</span>
    .*?
    <time\s+datetime="(?P<datetime>[^"]+)">
    .*?
    <div\s+class="tweet-text">(?P<tweet_text>.*?)</div>
    .*?
    <span\s+class="like-count">(?P<like_count>.*?)</span>
    .*?
    <span\s+class="rt-count">(?P<rt_count>.*?)</span>
    .*?
    <span\s+class="reply-count">(?P<reply_count>.*?)</span>
''', re.VERBOSE | re.DOTALL)

for html in html_snippets:
  m = tweet_pattern.search(html)
  if m:
    print(m.groupdict())

{'tweet_id': '1234567890', 'username': 'budi_nlp', 'display_name': 'Budi NLP', 'datetime': '2024-03-15T09:23:41.000Z', 'tweet_text': 'gila modelnya overfit parah, loss 0.001 tapi acc test cuma 55% XD #NLP', 'like_count': '1.2K', 'rt_count': '342', 'reply_count': '89'}
{'tweet_id': '9876543210', 'username': 'sari_ml99', 'display_name': 'Sari &amp; ML', 'datetime': '2024-03-15T11:05:17.000Z', 'tweet_text': '@budi_nlp regularisasi bro, pake dropout + early stopping &#128514;', 'like_count': '892', 'rt_count': '201', 'reply_count': '47'}


## Soal 7 [5 poin] — Deteksi Sinyal Berita Palsu (Hoax Signals)


In [ ]:
import re

hoax_candidates = [
    "BREAKING!! BOCORAN dari sumber terpercaya: pemerintah akan BLOKIR semua VPN SEKARANG JUGA!!!",
    "kata orang-orang, katanya sih model AI baru bisa baca pikiran, sebarkan sebelum dihapus!!",
    "update paper terbaru dari @yannlecun soal energy-based models, worth reading #ML",
    "TERBUKTI VIRAL: menurut sumber, vaksin mengandung chip 5G. jangan sampai terlambat!!",
    "NLP CONFERENCE DEADLINE EXTENDED sampai April 2024, cek situsnya guys #NLProc",
    "SEGERA SEBARKAN sebelum dihapus: kata mereka besok aplikasi X akan shutdown permanen!!!",
    ]

HOAX_PATTERNS = {
    'sensational': re.compile(r'\b(BREAKING|BOCORAN|VIRAL|TERBUKTI)\b', re.IGNORECASE),
    'urgency': re.compile(r'\b(sekarang juga|segera sebarkan|sebarkan sebelum dihapus|sebelum dihapus|jangan sampai terlambat)\b', re.IGNORECASE),
    'vague_source': re.compile(r'\b(kata|katanya|menurut|sumber)\s+(orang-orang|mereka|sumber|terpercaya)?\b', re.IGNORECASE),
    'capslock_spam': re.compile(r'(?:\b[A-Z]{2,}\b\s*){4,}'),
    }

def hoax_score(tweet: str) -> dict:
  signals = {k: bool(p.search(tweet)) for k, p in HOAX_PATTERNS.items()}
  signals['score'] = sum(signals.values())
  return signals

for tw in hoax_candidates:
  print(tw[:50], '->', hoax_score(tw))

BREAKING!! BOCORAN dari sumber terpercaya: pemerin -> {'sensational': True, 'urgency': True, 'vague_source': True, 'capslock_spam': False, 'score': 3}
kata orang-orang, katanya sih model AI baru bisa b -> {'sensational': False, 'urgency': True, 'vague_source': True, 'capslock_spam': False, 'score': 2}
update paper terbaru dari @yannlecun soal energy-b -> {'sensational': False, 'urgency': False, 'vague_source': False, 'capslock_spam': False, 'score': 0}
TERBUKTI VIRAL: menurut sumber, vaksin mengandung  -> {'sensational': True, 'urgency': True, 'vague_source': True, 'capslock_spam': False, 'score': 3}
NLP CONFERENCE DEADLINE EXTENDED sampai April 2024 -> {'sensational': False, 'urgency': False, 'vague_source': False, 'capslock_spam': True, 'score': 1}
SEGERA SEBARKAN sebelum dihapus: kata mereka besok -> {'sensational': False, 'urgency': True, 'vague_source': True, 'capslock_spam': False, 'score': 2}


## Soal 8 [5 poin] — Ekstraktor Topik dari Thread Twitter

In [ ]:
import re

thread_tweets = [
    "1/ Oke gas bahas kenapa GPT-4 masih kalah sama manusia di beberapa task NLP #thread",
    "2/ Pertama, accuracy-nya di benchmark DROP cuma 82.3% vs manusia 96.4%, gap gede bgt",
    "3/ Kedua, BERT-large juga struggling di long-context > 512 token, ini known issue",
    "4/ Solusi? @sari_ml bilang pake T5 + retrieval augmentation. tested di squad2.0 oke",
    "5/ TL;DR: LLM bagus tapi bukan silver bullet. fine-tuning + domain data tetep penting!",
    "--- bonus: paper referensinya ada di https://t.co/threadref123 cc: @budi_nlp @andi_ai",
    ]

THREAD_NUM = re.compile(r'^(\d+)[/.]\s+') # deteksi penomoran
AI_MODEL = re.compile(r'\b(GPT-[\d.]+o?|BERT(?:-\w+)?|T5|LLaMA-?\d*|Gemini|Claude|Mistral)\b') # nama model AI (GPT-4, BERT, T5, dll)
NUMBERS = re.compile(r'\b\d+(?:\.\d+)?%?|\b\d+(?:\.\d+)?\s*(?:juta|ribu|jt|rb)\b') # angka & persentase
TECH_TERM = re.compile(r'\b(NLP|benchmark|accuracy|long-context|token|retrieval augmentation|fine-tuning|domain data|LLM|DROP|squad2\.0)\b', re.IGNORECASE) # istilah teknis NLP

def parse_thread(tweets: list) -> dict:
  result = {'is_thread': False, 'content': [], 'entities': {}}

  # implementasi di sini
  numbered_tweets = []

  for tweet in tweets:
    m = THREAD_NUM.match(tweet)
    if m:
      number = int(m.group(1))
      content = THREAD_NUM.sub('', tweet)

      numbered_tweets.append({
          'number': number,
          'text': content})

  if numbered_tweets:
    result['is_thread'] = True
    numbered_tweets = sorted(numbered_tweets, key=lambda x: x['number'])

    result['content'] = numbered_tweets
    result['merged_text'] = ' '.join(item['text'] for item in numbered_tweets)

    merged = result['merged_text']

    result['entities'] = {
        'ai_models': AI_MODEL.findall(merged),
        'numbers': NUMBERS.findall(merged),
        'tech_terms': TECH_TERM.findall(merged)
    }

  return result

parsed = parse_thread(thread_tweets)

print('Thread?:', parsed['is_thread'])
print('Entities:', parsed['entities'])

Thread?: True
Entities: {'ai_models': ['GPT-4', 'BERT-large', 'T5'], 'numbers': ['4', '82.3%', '96.4%', '512', '0'], 'tech_terms': ['NLP', 'accuracy', 'benchmark', 'DROP', 'long-context', 'token', 'retrieval augmentation', 'squad2.0', 'LLM', 'fine-tuning', 'domain data']}


## Soal 9 [5 poin] — Sentiment Pre-processing: Emoji & Emoticon

In [ ]:
import re

sentiment_tweets = [
"model gw akhirnya converge juga wkwk seneng bgt!! :-D hasilnya bagus poll #blessed",
"overfit lagi overfit lagi :( gak ngerti kenapa loss naik terus ya Allah huhu",
"kagak ngerti deh sama paper ini, dijelasin juga ga masuk-masuk :/ pusing abis",
"gak mungkin dong akurasi 99% di real data, pasti ada data leakage :))) curiga",
"suka banget sama @budi_nlp tugasnya, kreatif parah & ga boring sama sekali bet!",
]

EMOTICONS = {
r':-?[)D]|:D|=\)': 'POSITIVE_EMOJI',
r':-?[/(|]|:[\\/]': 'NEGATIVE_EMOJI',
r':-?~|:/': 'NEUTRAL_EMOJI',
}

POS_UNICODE = re.compile(r'[😀😃😄😁😂🤣😊😍👍❤️]')
NEG_UNICODE = re.compile(r'[😢😭😡😠😞😔👎]')

INTENSIFIER = re.compile(r'\b(banget|bgt|poll|abis|parah|bet)\b', re.IGNORECASE)
NEGATION = re.compile(r'\b(ga|gak|tidak|kagak|bukan|gak mungkin|ga ada)\b', re.IGNORECASE)

def preprocess_sentiment(tweet: str) -> dict:
  # return dict: cleaned_text, emoticons, intensifiers, negations
  cleaned_text = tweet

  emoticons_found = []

  for pattern, label in EMOTICONS.items():
    found = re.findall(pattern, cleaned_text)
    if found:
      emoticons_found.extend([label] * len(found))
      cleaned_text = re.sub(pattern, label, cleaned_text)

  pos_emoji = POS_UNICODE.findall(cleaned_text)
  neg_emoji = NEG_UNICODE.findall(cleaned_text)

  cleaned_text = POS_UNICODE.sub('EMOJI_POS', cleaned_text)
  cleaned_text = NEG_UNICODE.sub('EMOJI_NEG', cleaned_text)

  intensifiers = INTENSIFIER.findall(cleaned_text)
  negations = NEGATION.findall(cleaned_text)

  return {
      'cleaned_text': cleaned_text,
      'emoticons': emoticons_found,
      'unicode_positive_emoji': pos_emoji,
      'unicode_negative_emoji': neg_emoji,
      'intensifiers': intensifiers,
      'negations': negations
  }

for tw in sentiment_tweets:
  result = preprocess_sentiment(tw)
  print(result)

{'cleaned_text': 'model gw akhirnya converge juga wkwk seneng bgt!! POSITIVE_EMOJI hasilnya bagus poll #blessed', 'emoticons': ['POSITIVE_EMOJI'], 'unicode_positive_emoji': [], 'unicode_negative_emoji': [], 'intensifiers': ['bgt', 'poll'], 'negations': []}
{'cleaned_text': 'overfit lagi overfit lagi NEGATIVE_EMOJI gak ngerti kenapa loss naik terus ya Allah huhu', 'emoticons': ['NEGATIVE_EMOJI'], 'unicode_positive_emoji': [], 'unicode_negative_emoji': [], 'intensifiers': [], 'negations': ['gak']}
{'cleaned_text': 'kagak ngerti deh sama paper ini, dijelasin juga ga masuk-masuk NEGATIVE_EMOJI pusing abis', 'emoticons': ['NEGATIVE_EMOJI'], 'unicode_positive_emoji': [], 'unicode_negative_emoji': [], 'intensifiers': ['abis'], 'negations': ['kagak', 'ga']}
{'cleaned_text': 'gak mungkin dong akurasi 99% di real data, pasti ada data leakage POSITIVE_EMOJI)) curiga', 'emoticons': ['POSITIVE_EMOJI'], 'unicode_positive_emoji': [], 'unicode_negative_emoji': [], 'intensifiers': [], 'negations': ['ga

## Soal 10 [5 poin] — Twitter Bot Detection Pattern

In [ ]:
import re

bot_data = [
    {'user': 'xK9mNpQ2024', 'text': 'Follow @promo_akun untuk info menarik!! #follow #followback #f4f #FF #teamfollowback #gain'},
    {'user': 'budi_nlp', 'text': 'overfit parah nih modelnya wkwk, loss terus naik padahal data udh banyak'},
    {'user': 'User83920174', 'text': 'Dapatkan penghasilan Rp500.000/hari!! Hubungi kami #kerja #online #uang #passive #income #cuan'},
    {'user': 'sari_ml99', 'text': 'setuju sama @budi_nlp, regularisasi emg penting bgt buat generalization'},
    {'user': 'PromoBot7821', 'text': 'Promo hari ini: diskon 90%!! #promo #sale #murah #diskon #belanja #viral #trending'},
    ]

BOT_USERNAME = re.compile(r'^(?:[A-Za-z]{2,10}\d{4,}|[A-Za-z0-9]{10,})$') # pattern username bot
HASHTAG_SPAM = re.compile(r'#[A-Za-z0-9_]+') # deteksi hashtag > 5
PROMO_LANGUAGE = re.compile(r'\b(diskon|promo|gratis|penghasilan|cuan|rupiah|hubungi|followback|f4f)\b', re.IGNORECASE)

def bot_score(user: str, text: str) -> dict:
  signals = {
      'bot_username': bool(BOT_USERNAME.match(user)),
      'hashtag_spam': len(HASHTAG_SPAM.findall(text)) > 5,
      'promo_language': bool(PROMO_LANGUAGE.search(text)),
      }

  signals['is_likely_bot'] = sum(signals.values()) >= 2

  return signals

for item in bot_data:
  print(item['user'], '->', bot_score(item['user'], item['text']))

xK9mNpQ2024 -> {'bot_username': True, 'hashtag_spam': True, 'promo_language': True, 'is_likely_bot': True}
budi_nlp -> {'bot_username': False, 'hashtag_spam': False, 'promo_language': False, 'is_likely_bot': False}
User83920174 -> {'bot_username': True, 'hashtag_spam': True, 'promo_language': True, 'is_likely_bot': True}
sari_ml99 -> {'bot_username': False, 'hashtag_spam': False, 'promo_language': False, 'is_likely_bot': False}
PromoBot7821 -> {'bot_username': True, 'hashtag_spam': True, 'promo_language': True, 'is_likely_bot': True}


# BAGIAN B — Analisis Regex [Soal 11 – 20]

## Soal 11 [5 poin] — Regex Cleaner Tweet Berantakan

In [ ]:
#import re

#tweets = [
    #"follow gw ya guys!! @budi_nlp @sari99 cc @andi #NLP haha http://t.co/abc wkwk :D",
    #"RT @user: &lt;b&gt;breaking&lt;/b&gt; overfit parah bgt 99.9%% akurasinya LOL!!!!!",
    #]

#cleaner = re.compile(r'''
#https?://\S+ # URL
#| &(?:[a-z]+|\#\d+); # HTML entity
#| <[^>]+> # HTML tag
#| RT\s+@\w+:\s* # retweet prefix
#| @\w+ # mention
#| \#\w+ # hashtag
#| [^\w\s] # non-word non-space
#''', re.VERBOSE | re.IGNORECASE)

#for tw in tweets:
    #result = cleaner.sub(' ', tw)
    #result = re.sub(r'\s{2,}', ' ', result).strip()
    #print(repr(result))

Jawab pertanyaan berikut:

**a) Jelaskan kenapa urutan alternation (|) dalam pattern ini penting — apa yang terjadi kalau urutan dibalik?**

Jawab: Karena regex bekerja dengan prinsip “first match wins”. Artinya regex akan mencoba dari kiri ke kanan. Begitu ada yang match → langsung dipakai, tidak cek yang lain

Contoh dampaknya: Kalau [^\w\s] (hapus semua simbol) diletakkan di awal
→ maka @mention, #hashtag, bahkan URL bisa ikut terpotong sebelum sempat dikenali sebagai pola khusus

Jadi urutan dibuat dari pola spesifik (URL, HTML, mention, hashtag) ke pola paling umum (simbol)

**b) Apa yang dimatch &(?:[a-z]+|\#\d+); — berikan 3 contoh string yang match.**

Jawab: Artinya:

- & → diawali tanda &
- (?: ... ) → isi bisa:
  - [a-z]+ → huruf (named entity)
  - \#\d+ → angka (numeric entity)
- ; → diakhiri titik koma

Contoh yang match:

&amp;
&lt;
&#128514;

**c) Kenapa [^\w\s] diletakkan paling terakhir?**

Jawab: Karena ini adalah “catch-all”: yaitu menangkap semua karakter selain huruf & spasi

Kalau diletakkan di depan, simbol penting seperti @, #, : akan langsung dihapus
sehingga: @user tidak dikenali sebagai mention dan #NLP tidak dikenali sebagai hashtag

Jadi harus di paling akhir agar pola penting diproses dulu baru sisa “noise” dibersihkan

**d) Prediksi output (repr) untuk kedua tweet.**

Jawab:

Tweet 1: Semua @mention, #hashtag, URL, dan simbol (!!, :D) dihapus.

Hasil: 'follow gw ya guys cc haha wkwk D'

Tweet 1: Yang dihapus adalah RT @user:, HTML entity &lt; &gt;, HTML tag <b>, simbol !!!!!, %%

Hasil: 'b breaking b overfit parah bgt 99 9 akurasinya LOL'

## Soal 12 [5 poin] — Deteksi Pola Engagement Rate

In [ ]:
#import re

#tweet_stats = [
    #"1.2K likes, 342 RTs, 89 replies — engagement rate: 14.7%",
    #"892 likes, 201 RTs — total reach: 15,420 impressions (est.)",
    #"10.3K likes | 5.2K RTs | 2.1K replies | 102K impressions",
    #"Likes: 0, RT: 0, Reply: 1 — ratio jelek banget wkwk",
    #]

#NUM_PATTERN = re.compile(
    #r'(?P<value>\d+(?:[.,]\d+)?)\s*(?P<unit>K|M|B)?'
    #r'\s*(?P<metric>likes?|RTs?|retweets?|replies?|impressions?|reach)',
    #re.IGNORECASE
    #)

#for stat in tweet_stats:
  #for m in NUM_PATTERN.finditer(stat):
    #val = float(m.group('value').replace(',', '.'))
    #mult = {'K': 1000, 'M': 1_000_000, 'B': 1_000_000_000}
    #actual = val * mult.get(m.group('unit') or '', 1)
    #print(f"{m.group('metric'):12} = {actual:>12,.0f}")
  #print('---')

Jawab pertanyaan berikut:

**a) Jelaskan \d+(?:[.,]\d+)? — kenapa pakai (?:...) bukan (...)?**

Jawab: Regex tersebut digunakan untuk menangkap angka, baik angka bulat maupun angka desimal.

- \d+ berarti satu atau lebih digit angka.
- (?:[.,]\d+)? berarti bagian desimal yang sifatnya opsional.
- [.,] berarti pemisah desimal boleh berupa titik atau koma.
- \d+ setelahnya berarti angka setelah titik atau koma.

Contoh yang cocok: 1, 1.2, 10.3, 2,5.

Digunakan (?:...) karena bagian tersebut hanya diperlukan untuk mengelompokkan pola, bukan untuk disimpan sebagai group tersendiri. Jadi (?:...) disebut non-capturing group.

**b) Apa yang terjadi kalau unit-nya None (tidak ada K/M/B)? Kenapa m.group('unit') or '' diperlukan?**

Jawab: Jika angka tidak memiliki unit seperti K, M, atau B, maka "m.group('unit')" akan menghasilkan None.

Karena itu digunakan "m.group('unit') or ''". Tujuannya agar jika unit tidak ada, nilainya diganti menjadi string kosong ''. Dengan begitu, kode "mult.get(m.group('unit') or '', 1)" akan mengembalikan nilai default 1.

Contohnya:

892 likes

Tidak memiliki unit, sehingga 892 × 1 = 892

**c) Untuk tweet ke-3 '10.3K likes | 5.2K RTs | ...', berapa baris output yang dihasilkan?**

Jawab: Ada 4 metrik yang cocok dengan regex, yaitu:

- 10.3K likes
- 5.2K RTs
- 2.1K replies
- 102K impressions

Jadi, tweet ke-3 menghasilkan 4 baris output.

**d) Prediksi nilai actual untuk '1.2K likes' dan '102K impressions'.**

Jawab:

- Untuk: 1.2K likes

Nilainya adalah: 1.2 × 1000 = 1200

Jadi actual-nya adalah 1,200.

- Untuk: 102K impressions

Nilainya adalah: 102 × 1000 = 102000

Jadi actual-nya adalah 102,000.

## Soal 13 [5 poin] — Username Extractor dengan Lookaround

In [ ]:
#import re

#texts = [
    #"gw udh follow @budi_nlp dari 2019, email-nya budi@nlplab.ac.id btw",
    #"contact: admin@twitter-tools.io atau via DM ke @sari_ml99 aja",
    #"bukan @@double_at dan bukan email@mention tapi @real_user bisa",
    #"copyright 2024@company.com dan @@weirduser dan @valid123",
    #]

#MENTION = re.compile(r'(?<![.\w@])@([A-Za-z]\w{0,14})(?!\.\w)')

#for text in texts:
  #found = MENTION.findall(text)
  #print(f'Mentions: {found}')

Jawab pertanyaan berikut:

**a) Jelaskan lookbehind (?<![.\w@]) — karakter apa yang TIDAK boleh ada sebelum @?**

Jawab: (?<![.\w@]) adalah negative lookbehind, artinya karakter sebelum tanda @ tidak boleh berupa:

- titik (.)
- huruf/angka/underscore (\w)
- tanda (@)

Tujuannya agar regex tidak salah mendeteksi email atau double mention seperti @@user.

**b) Kenapa pattern username [A-Za-z]\w{0,14} — apa batasan panjang dan karakter pertamanya?**

Jawab: Karena,

- [A-Za-z] berarti karakter pertama username harus berupa huruf.
- \w{0,14} berarti setelah huruf pertama boleh diikuti huruf, angka, atau underscore sebanyak 0 sampai 14 karakter.

Jadi panjang username yang bisa terdeteksi adalah 1 sampai 15 karakter.

**c) Jelaskan lookahead (?!\.\w) — ini mencegah match apa?**

Jawab: (?!\.\w) adalah negative lookahead, artinya setelah username tidak boleh ada titik yang diikuti huruf/angka.

Tujuannya untuk mencegah regex mendeteksi bagian email sebagai mention.

Contoh yang dicegah: budi@nlplab.ac.id

Bagian @nlplab tidak dianggap mention karena setelahnya ada .ac.id

**d) Prediksi output untuk setiap teks (list mention yang ditemukan).**

Jawab:

- Mentions: ['budi_nlp']

Tweet pertama hanya mengambil @budi_nlp, sedangkan budi@nlplab.ac.id tidak diambil karena itu email.

- Mentions: ['sari_ml99']

Tweet kedua hanya mengambil @sari_ml99, sedangkan admin@twitter-tools.io tidak diambil karena email.

- Mentions: ['real_user']

Tweet ketiga hanya mengambil @real_user, sedangkan @@double_at dan email@mention tidak valid.

- Mentions: ['valid123']

Tweet keempat hanya mengambil @valid123, sedangkan 2024@company.com dan @@weirduser tidak valid.

## Soal 14 [5 poin] — Thread Numbering Detector

In [ ]:
#import re

#thread_examples = [
    #"1/ oke gas bahas kenapa model ini gagal total di production",
    #"2/10 lanjut dari tadi, ternyata data imbalance parah banget",
    #"thread: 3. ini poin ketiga yang sering dilupain developer",
    #"gak ada nomornya ini bukan thread biasa aja",
    #"(4) poin keempat: evaluasi model itu beda sama accuracy doang",
    #"5 - terakhir: jangan lupa validate di out-of-distribution data ya",
    #]

#THREAD_NUM = re.compile(
    #r'^(?:\()?(\d{1,2})(?:\)|[/.\-]|\s*(?:of|dari|\/)?\s*\d{0,2})\s',
    #re.IGNORECASE
    #)

#for ex in thread_examples:
  #m = THREAD_NUM.match(ex)
  #if m:
    #print(f'Thread #{m.group(1):>2}: {ex[:50]}')
  #else:
    #print(f'Non-thread : {ex[:50]}')

Jawab pertanyaan berikut:

**a) Jelaskan (?:\()? — apa yang diizinkan di awal sebelum angka?**

Jawab: Bagian ini berarti regex mengizinkan adanya tanda kurung buka ( sebelum angka.

Contoh yang bisa match: (4)

Tanda ?: menunjukkan bahwa grup tersebut non-capturing group, jadi tidak disimpan sebagai hasil grup.

**b) Apa arti** (?:\)|[/.\-]|\s*(?:of|dari|\/)? \s*\d{0,2}) **— sebutkan semua format yang bisa match.**

Jawab: Bagian ini digunakan untuk mendeteksi format setelah angka thread.

Format yang bisa match yaitu:

- Tanda kurung tutup: (4)
- Slash: 1/
- Titik: 3.
- Strip: 5 -
- Format dengan total thread: 2/10
- Format seperti 2 of 10 atau 2 dari 10

**c) Kenapa dipakai re.match() bukan re.search() untuk pattern ini?**

Jawab: Karena nomor thread biasanya harus berada di awal teks.

re.match() hanya mencari pola dari awal string, sehingga lebih tepat untuk mendeteksi tweet yang memang diawali nomor thread.

Kalau memakai re.search(), angka di tengah kalimat bisa ikut terdeteksi, padahal belum tentu itu nomor thread.

**d) Prediksi output (Thread #N atau Non-thread) untuk setiap string.**

Jawab:

Thread # 1: 1/ oke gas bahas kenapa model ini gagal total di p
Thread # 2: 2/10 lanjut dari tadi, ternyata data imbalance par
Non-thread : thread: 3. ini poin ketiga yang sering dilupain de
Non-thread : gak ada nomornya ini bukan thread biasa aja
Thread # 4: (4) poin keempat: evaluasi model itu beda sama acc
Thread # 5: 5 - terakhir: jangan lupa validate di out-of-distr

## Soal 15 [5 poin] — Alay Score Calculator

In [ ]:
#import re

#alay_tweets = [
    #"haappyyy bgt modelnya WORK akhirnya!! wkwkwkwkwk",
    #"Saya tidak mengerti mengapa model ini tidak bekerja.",
    #"gak nyangkaaaaa loss-nya 0.0001 GILAAAAA bgt pollll!!!!!",
    #"mantap jiwa bro, model lo emg the best deh fr fr fr",
    #]

#patterns = {
    #'elongasi': re.compile(r'(.)\1{2,}'),
    #'capslock': re.compile(r'\b[A-Z]{3,}\b'),
    #'wkwk_chain': re.compile(r'(?:wk){3,}', re.IGNORECASE),
    #'exclaim': re.compile(r'!{2,}'),
    #'slang_reps': re.compile(r'\b(\w+)\b(?:\s+\1){2,}', re.IGNORECASE),
    #}

#for tw in alay_tweets:
  #scores = {k: len(p.findall(tw)) for k, p in patterns.items()}
  #total = sum(scores.values())
  #label = 'SANGAT ALAY' if total > 4 else 'ALAY' if total > 1 else 'FORMAL'
  #print(f'{label:12} (score={total}): {tw[:45]}')

Jawab pertanyaan berikut:

**a) Jelaskan (.)\1{2,} — karakter apa yang dimatch dan apa fungsi \1?**

Jawab:

- (.) menangkap satu karakter apa saja
- \1 mengacu ke karakter yang sama dengan hasil tangkapan pertama
- {2,} berarti karakter itu muncul lagi minimal 2 kali

Jadi pattern ini mendeteksi karakter yang muncul minimal 3 kali berturut-turut.

Contoh match:

- aaa
- yyyy
- !!!!!

**b) Pattern (?:wk){3,} — mengapa non-capturing group dipakai di sini?**

Jawab: Pattern ini mendeteksi pengulangan wk minimal 3 kali.

Contoh match:

- wkwkwk
- wkwkwkwk

(?:...) dipakai karena kita hanya butuh mendeteksi polanya, bukan menyimpan isi grupnya. Jadi lebih rapi dan tidak menambah hasil tangkapan yang tidak perlu.

**c) Jelaskan (\w+)\b(?:\s+\1){2,} — ini mendeteksi pola apa? Berikan contoh match-nya.**

Jawab: Pattern ini mendeteksi kata yang diulang minimal 3 kali.

- (\w+) → menangkap satu kata
- \b → batas kata
- (?:\s+\1){2,} → kata yang sama muncul lagi minimal 2 kali setelah spasi

Contoh match:

- fr fr fr
- ha ha ha
- iya iya iya

**d) Prediksi label dan score untuk setiap tweet.**

Jawab:

**- Tweet: haappyyy bgt modelnya WORK akhirnya!! wkwkwkwkwk**

Temuan:

- Elongasi (huruf berulang berlebihan): yyy → 1
- Capslock (kata huruf kapital semua): WORK → 1
- Rantai wkwk: wkwkwkwkwk → 1
- Tanda seru berlebihan: !! → 1
- Pengulangan kata: 0

Total score = 4

ALAY         (score=4): haappyyy bgt modelnya WORK akhirnya!! wkwkwkw

**- Tweet: Saya tidak mengerti mengapa model ini tidak bekerja.**

Temuan:

- tidak ada pola alay

Total score = 0

FORMAL       (score=0): Saya tidak mengerti mengapa model ini tidak b

**- Tweet: gak nyangkaaaaa loss-nya 0.0001 GILAAAAA bgt pollll!!!!!**

Temuan:

- Elongasi (huruf berulang berlebihan): aaaaa, AAAAA, llll, !!!!! → 4
- Capslock (kata huruf kapital semua): GILAAAAA → 1
- Tanda seru berlebihan: !!!!! → 1
- Rantai wkwk: 0
- Pengulangan kata: 0

Total score = 6

SANGAT ALAY  (score=6): gak nyangkaaaaa loss-nya 0.0001 GILAAAAA bgt

**- Tweet: mantap jiwa bro, model lo emg the best deh fr fr fr**

Temuan:

- Elongasi (huruf berulang berlebihan): 0
- Capslock (kata huruf kapital semua): 0
- Tanda seru berlebihan: 0
- Rantai wkwk: 0
- Pengulangan kata: fr fr fr → 1

Total score = 1

FORMAL       (score=1): mantap jiwa bro, model lo emg the best deh fr

## Soal 16 [5 poin] — Twitter Bio Parser

In [ ]:
#import re

#bios = [
    #"ML Engineer @GojekTech | PhD @ui_official | ex-@google | tweets = opini pribadi",
    #"NLP Researcher | Suka kopi & ngoding | DM for collab | she/her | Jakarta, ID",
    #"AI enthusiast | Data Scientist at Tokopedia | ITB 2019 | link: linktr.ee/aiuser",
    #"random guy yang suka overthinking soal LLMs | not affiliated w/ any company lol",
    #]

#BIO_PATTERN = re.compile(r'''
#@(?P<org>[A-Za-z]\w+) # org mention
#| (?P<role>(?:ML|NLP|AI|Data)\s+\w+) # tech role
#| (?P<uni>(?:PhD|S1|S2|@\w+)\s*(?:@\w+)?) # education
#| (?P<loc>[A-Z][a-z]+(?:,\s*[A-Z]{2,3})?) # location
#''', re.VERBOSE)

#for bio in bios:
  #entities = {}
  #for m in BIO_PATTERN.finditer(bio):
    #for k, v in m.groupdict().items():
      #if v:
        #entities.setdefault(k, []).append(v.strip())
  #print(entities)

Jawab pertanyaan berikut:

**a) Jelaskan perbedaan antara (?P<org>...) dan (?P<role>...) — apa yang membedakan keduanya?**

Jawab:

(?P<org>...) adalah named group untuk menangkap organisasi berbentuk mention.

Contoh:

- @GojekTech
- @ui_official
- @google

Sedangkan (?P<role>...) menangkap role/pekerjaan yang diawali istilah teknologi seperti:

- ML Engineer
- NLP Researcher
- AI enthusiast
- Data Scientist

**b) Pattern loc:** [A-Z][a-z]+(?:,\s*[A-Z]{2,3})? **— kota/negara seperti apa yang bisa match?**

Jawab: Arti dari pattern loc tersebut adalah

- [A-Z] → diawali huruf kapital
- [a-z]+ → diikuti huruf kecil
- (?:,\s*[A-Z]{2,3})? → boleh diikuti kode negara 2–3 huruf

Contoh yang bisa match:

- Jakarta
- Jakarta, ID
- Bandung, ID
- Tokyo, JPN

**c) m.groupdict() mengembalikan apa? Kenapa perlu cek if v?**

Jawab: m.groupdict() mengembalikan dictionary dari semua named group.

Contoh bentuknya:

{
  'org': '@GojekTech',
  'role': None,
  'uni': None,
  'loc': None
}

Perlu cek if v karena dalam satu match, hanya salah satu group yang biasanya terisi. Group lain bernilai None. Jadi if v dipakai agar hanya hasil yang benar-benar match yang dimasukkan ke entities.

**d) Prediksi output (dict entities) untuk bio pertama dan ketiga.**

Jawab:

**- Bio pertama**

{'role': ['ML Engineer'], 'org': ['GojekTech', 'google'], 'uni': ['PhD @ui_official']}

Penjelasan:

- ML Engineer masuk role
- @GojekTech dan @google masuk org, tapi hasilnya hanya GojekTech, google karena group org tidak menyimpan tanda @
- PhD @ui_official masuk uni
- @ui_official tidak masuk org karena sudah tertangkap sebagai bagian uni

**- Bio ketiga**

{'role': ['AI enthusiast', 'Data Scientist'], 'loc': ['Tokopedia']}

Penjelasan:

- AI enthusiast dan Data Scientist masuk role
- Tokopedia salah terbaca sebagai loc karena formatnya mirip lokasi: huruf kapital di awal lalu huruf kecil

## Soal 17 [5 poin] — Regex NER: Ekstrak Nama Model AI dari Tweet

In [ ]:
#import re

#tweets_ai = [
    #"GPT-4o vs Gemini-1.5-Pro buat summarization, mana lebih bagus? cc @openai @google",
    #"fine-tuning LLaMA-3-8B sama Mistral-7B-Instruct di dataset bahasa Indonesia, hasilnya???",
    #"BERT-base-multilingual-cased masih worth it di 2024? atau langsung pake XLM-RoBERTa-large?",
    #"nge-run Claude-3-Opus sama GPT-4-Turbo buat eval, Claude lebih hemat token kayaknya",
    #"gpt4 (tanpa strip) sama gemini pro juga harus kedetect ya ga cuma yang format resmi",
    #]

#MODEL_PATTERN = re.compile(
    #r'\b(?:'
    #r'GPT-?(?:4o?|3\.5)(?:-\w+)*'
    #r'|Gemini-?(?:\d+(?:\.\d+)?)?(?:-\w+)*'
    #r'|(?:LLaMA|Llama)-?(?:\d+(?:-\d+[BbMm])?)?(?:-\w+)*'
    #r'|(?:BERT|RoBERTa|XLM-RoBERTa)(?:-\w+)*'
    #r'|Mistral-\d+[Bb](?:-\w+)*'
    #r'|Claude-\d+(?:-\w+)*'
    #r')\b',
    #re.IGNORECASE
    #)

#for tw in tweets_ai:
  #models = MODEL_PATTERN.findall(tw)
  #print(f'Models found: {models}')

Jawab pertanyaan berikut:

**a) Jelaskan GPT**-?(?:4o?|3\.5)(?:-\w+)* **— berikan 4 contoh string yang match dan 2 yang tidak.**

Jawab:

Artinya:

- GPT → harus diawali tulisan GPT
- -? → tanda strip - boleh ada atau tidak
- (?:4o?|3\.5) → versinya boleh:
  - 4
  - 4o
  - 3.5
- (?:-\w+)* → boleh ada tambahan setelah strip, misalnya -Turbo

Contoh yang match:

- GPT-4
- GPT-4o
- GPT4
- GPT-4-Turbo

Contoh yang tidak match:

- GPT-5
- GPT-2

**b) Kenapa re.IGNORECASE dipakai — implikasinya apa untuk matching 'gpt4' vs 'GPT-4'?**

Jawab: re.IGNORECASE membuat regex tidak membedakan huruf besar dan kecil.

Jadi:

- GPT-4
- gpt-4
- gpt4
- GpT4

semuanya tetap bisa terdeteksi.

Implikasinya, gpt4 di tweet terakhir tetap match meskipun ditulis huruf kecil.

**c) Apa bedanya \d+[BbMm] dengan \d+(?:\.\d+)? dalam konteks nama model?**

Jawab:

\d+[BbMm] → dipakai untuk ukuran model, seperti:

- 7B
- 8B
- 70B

Artinya angka diikuti huruf B, b, M, atau m dan biasanya menunjukkan jumlah parameter model.

\d+(?:\.\d+)? → dipakai untuk versi angka, misalnya:

- 1.5
- 3
- 4

Artinya angka boleh berupa bilangan bulat atau desimal.

**d) Prediksi findall() untuk tweet pertama dan kedua.**

Jawab:

**- Tweet pertama: GPT-4o vs Gemini-1.5-Pro buat summarization, mana lebih bagus? cc @openai @google**

Prediksi:

Models found: ['GPT-4o', 'Gemini-1.5-Pro']

**- Tweet kedua: fine-tuning LLaMA-3-8B sama Mistral-7B-Instruct di dataset bahasa Indonesia, hasilnya???**

Prediksi:

Models found: ['LLaMA-3-8B', 'Mistral-7B-Instruct']

## Soal 18 [5 poin] — Spam Reply Pattern Detector

In [ ]:
#import re

#replies = [
    #"@budi_nlp follow back dong kak!! udah follow nih #followback #FF",
    #"@sari_ml wih bagus bgt papernya, bisa share dataset-nya ga?",
    #"@andi_ai cek juga akun gw ya! ada konten NLP bagus kok dijamin!!",
    #"@target mampir ke bio gw ada link menarik buat kalian semua!!",
    #"@nlp_indo pertanyaan soal tokenisasi bahasa Jawa ada referensinya ga?",
    #]

#SPAM_SIGNALS = re.compile(r'''
#(?:follow\s*(?:back|me|gw|aku)) # follow bait
#| (?:cek\s+(?:bio|akun|profil)\s+gw) # profile bait
#| (?:(?:dijamin|pasti)\s+\w+!+) # false guarantee
#| (?:\#(?:follow|FF|f4f)\w*) # spam hashtag
#| (?:ada\s+link\s+\w+) # link bait
#''', re.VERBOSE | re.IGNORECASE)

#for reply in replies:
  #signals = SPAM_SIGNALS.findall(reply)
  #is_spam = len(signals) >= 1
  #print(f"{'SPAM' if is_spam else 'OK '} | signals={signals} | {reply[:40]}")

Jawab pertanyaan berikut:

**a) Jelaskan** (?:follow\s*(?:back|me|gw|aku)) **— kenapa** \s* **bukan** \s+?

Jawab:

Artinya:

- follow → mencari kata “follow”
- \s* → boleh ada spasi atau tidak ada spasi
- (?:back|me|gw|aku) → setelah follow boleh berupa:
  - back
  - me
  - gw
  - aku

Contoh yang match:

- follow back
- followback
- follow gw
- follow aku

Pakai \s* bukan \s+ karena \s* membuat regex bisa menangkap dua bentuk:

- follow back
- followback

Kalau \s+, maka followback tidak akan match.

**b) Apa fungsi re.VERBOSE di sini — apa yang terjadi tanpa flag ini?**

Jawab: re.VERBOSE membuat regex bisa ditulis lebih rapi dengan:

- spasi
- baris baru
- komentar #

Tanpa re.VERBOSE, spasi dan komentar di dalam regex akan dianggap bagian dari pattern, sehingga regex bisa gagal atau hasilnya tidak sesuai.

**c) Pattern** (?:(?:dijamin|pasti)\s+\w+!+) **— berikan 2 contoh yang match dan 1 yang tidak.**

Jawab:

Artinya:

- (?:dijamin|pasti) → harus diawali kata dijamin atau pasti
- \s+ → harus ada minimal satu spasi
- \w+ → diikuti satu kata
- !+ → diakhiri satu atau lebih tanda seru

Contoh yang match:

- dijamin bagus!!
- pasti menang!

Contoh yang tidak match: dijamin bagus

Karena tidak ada tanda seru di akhir.

**d) Prediksi output (SPAM atau OK) beserta signals untuk setiap reply.**

Jawab:

**- Reply pertama: @budi_nlp follow back dong kak!! udah follow nih #followback #FF**

Prediksi output pertama:

Signals yang match:

['follow back', '#followback', '#FF']

Output:

SPAM | signals=['follow back', '#followback', '#FF'] | @budi_nlp follow back dong kak!! udah fo

**- Reply kedua: @sari_ml wih bagus bgt papernya, bisa share dataset-nya ga?**

Prediksi output kedua:

Tidak ada sinyal spam.

[]

Output:

OK  | signals=[] | @sari_ml wih bagus bgt papernya, bisa sh

**- Reply ketiga: @andi_ai cek juga akun gw ya! ada konten NLP bagus kok dijamin!!**

Prediksi output ketiga:

Tidak match *cek akun gw*, karena pattern-nya mengharuskan *cek akun gw*

sedangkan teksnya *cek juga akun gw*

Tidak match *dijamin!!*, karena pattern butuh *dijamin + kata + tanda seru*

Output:

OK  | signals=[] | @andi_ai cek juga akun gw ya! ada konten

**- Reply keempat: @target mampir ke bio gw ada link menarik buat kalian semua!!**

Prediksi output keempat:

Di kalimat "ada link menarik"

- Mengandung pola link bait
- Cocok dengan regex *ada link ...*

→ Jadi regex berhasil mendeteksi pola "link bait"

Output:

SPAM | signals=['ada link menarik'] | @target mampir ke bio gw ada link menarik

**- Reply kelima: @nlp_indo pertanyaan soal tokenisasi bahasa Jawa ada referensinya ga?**

Prediksi output kelima:

Tidak ada sinyal spam.

[]

Output:

OK  | signals=[] | @nlp_indo pertanyaan soal tokenisasi bahas

## Soal 19 [5 poin] — Temporal Expression Extractor dari Tweet

In [38]:
#import re

#temporal_tweets = [
    #"dataset ini dikumpulin dari Jan 2022 sampai Maret 2024, ada 2.3jt tweet",
    #"deadline submission besok jam 23:59 WIB, jgn sampe telat guys!!",
    #"paper ini ditulis 3 tahun lalu tapi masih relevan di 2024 fr",
    #"meeting zoom tiap Senin & Kamis jam 19.30-21.00 WIB mulai minggu depan",
    #"crawling data dari 01/01/2023 sd 31/12/2023, total 6 bulan effort",
    #]

#TEMPORAL = re.compile(r'''
#(?P<date_range>
#(?:Jan(?:uari)?|Feb(?:ruari)?|Mar(?:et)?|Apr(?:il)?|Mei|Jun(?:i)?|
#Jul(?:i)?|Agu(?:stus)?|Sep(?:tember)?|Okt(?:ober)?|Nov(?:ember)?|Des(?:ember)?)
#\s+\d{4}\s+(?:sampai|sd|s\.?d\.?|-)\s+
#(?:Jan(?:uari)?|Feb(?:ruari)?|Mar(?:et)?|Apr(?:il)?|Mei|Jun(?:i)?|
#Jul(?:i)?|Agu(?:stus)?|Sep(?:tember)?|Okt(?:ober)?|Nov(?:ember)?|Des(?:ember)?)
#\s+\d{4}
#)
#| (?P<time>\d{1,2}[.:]\d{2}(?:\s*-\s*\d{1,2}[.:]\d{2})?\s*(?:WIB|WITA|WIT)?)
#| (?P<relative>\d+\s+(?:tahun|bulan|minggu|hari|jam)\s+(?:lalu|yang lalu|ke depan|lagi))
#| (?P<numdate>\d{2}/\d{2}/\d{4}\s+(?:sd|sampai|-)\s+\d{2}/\d{2}/\d{4})
#''', re.VERBOSE | re.IGNORECASE)

#for tw in temporal_tweets:
  #for m in TEMPORAL.finditer(tw):
    #print(f' [{m.lastgroup}] "{m.group()}"')
  #print()

Jawab pertanyaan berikut:

**a) Jelaskan m.lastgroup — bagaimana cara kerjanya ketika ada multiple named groups?**

Jawab: m.lastgroup mengembalikan nama group terakhir yang berhasil match.

Karena regex ini memakai named group seperti:

- (?P<date_range>...)
- (?P<time>...)
- (?P<relative>...)
- (?P<numdate>...)

Maka ketika ada teks yang cocok, m.lastgroup memberi tahu kategori mana yang cocok.

Contoh: 23:59 WIB

Akan menghasilkan: time

**b) Pattern date_range memakai alternation panjang. Apa trade-off dibanding r'[A-Za-z]+\s+\d{4}'?**

Jawab: Pattern date_range lebih panjang karena menuliskan nama bulan satu per satu.

Kelebihannya:

- lebih akurat
- hanya menangkap nama bulan yang valid
- bisa menangkap format rentang tanggal seperti Jan 2022 sampai Maret 2024

Kekurangannya:

- lebih panjang
- lebih sulit dibaca
- perlu ditambah manual kalau ada variasi nama bulan lain

Kalau memakai *r'[A-Za-z]+\s+\d{4}'* regex jadi lebih simpel, tapi bisa menangkap kata yang bukan bulan, misalnya:

- deadline 2024
- paper 2023

**c) Untuk tweet ke-4, apakah '19.30-21.00 WIB' akan match pattern time? Jelaskan.**

Jawab: Ya, match.

Karena pattern time: *\d{1,2}[.:]\d{2}(?:\s*-\s*\d{1,2}[.:]\d{2})?\s*(?:WIB|WITA|WIT)?*

bisa membaca:

- 19.30 → jam pertama
- -21.00 → rentang jam opsional
- WIB → zona waktu opsional

Jadi: 19.30-21.00 WIB

akan terdeteksi sebagai kategori: time

**d) Prediksi semua output (lastgroup + match) untuk tweet pertama dan keempat.**

**- Tweet pertama**

Input: dataset ini dikumpulin dari Jan 2022 sampai Maret 2024, ada 2.3jt tweet

Prediksi Output:

[date_range] "Jan 2022 sampai Maret 2024"

**- Tweet keempat**

Input: meeting zoom tiap Senin & Kamis jam 19.30-21.00 WIB mulai minggu depan

Prediksi Output:

[time] "19.30-21.00 WIB"

→ minggu depan tidak match, karena pattern relative butuh format seperti 1 minggu ke depan, bukan hanya minggu depan.


## Soal 20 [5 poin] — Master Regex: Full Tweet NER Pipeline [BOSS LEVEL]

In [40]:
#import re

#raw_tweet = (
    #"RT @budi_nlp: &lt;b&gt;BREAKING&lt;/b&gt; GPT-4o vs LLaMA-3-70B benchmark "
    #"di IndoNLU 2024!! accuracy: 94.7% vs 89.3% (p&lt;0.05). "
    #"dataset: https://t.co/xDataset123 | paper: bit.ly/IndoNLU24 "
    #"cc @sari_ml99 @andi_ai #NLP #BahasaIndonesia #LLM wkwkwk gilaaaa bet!!"
    #)

#MASTER = re.compile(r'''
#(?P<rt_prefix>^RT\s+@\w+:\s*)
#| (?P<html_tag><[^>]+>)
#| (?P<html_entity>&(?:[a-z]+|\#\d+);)
#| (?P<url>https?://\S+|(?:bit\.ly|t\.co)/\S+)
#| (?P<ai_model>
#GPT-?(?:4o?|3\.5)(?:-\w+)*
#| LLaMA-?\d+(?:-\d+[BbMm])?(?:-\w+)*
#)
#| (?P<percentage>\d+(?:\.\d+)?%)
#| (?P<mention>(?<![.\w@])@[A-Za-z]\w{0,14})
#| (?P<hashtag>\#[A-Za-z]\w+)
#| (?P<stat_sig>p\s*(?:&lt;|<|&gt;|>)\s*0\.\d+)
#''', re.VERBOSE | re.IGNORECASE | re.MULTILINE)

#entities = {}
#for m in MASTER.finditer(raw_tweet):
  #t = m.lastgroup
  #entities.setdefault(t, []).append(m.group())

#for entity_type, values in entities.items():
  #print(f'{entity_type:15}: {values}')

Jawab pertanyaan berikut:

**a) Pattern ini menggunakan alternation dengan 9 named groups. Jelaskan konsep 'first match wins' — contohkan dengan 2 group yang bisa overlap.**

Jawab:

Dalam regex dengan banyak alternation |, Python akan mencoba pola dari kiri ke kanan. Pola yang cocok lebih dulu akan dipakai.

Contoh overlap:

- RT @budi_nlp:
Bisa saja bagian @budi_nlp cocok sebagai mention, tetapi karena rt_prefix ditulis lebih dulu, maka seluruh RT @budi_nlp: masuk sebagai rt_prefix.
- p&lt;0.05
Di dalamnya ada &lt; yang bisa cocok sebagai html_entity, tetapi karena dari awal huruf p pola stat_sig cocok, maka semuanya masuk sebagai stat_sig.

**b) html_entity pattern: &(?:[a-z]+|\#\d+); — kenapa ada dua alternatif? Berikan contoh masing-masing.**

Jawab:

HTML entity punya dua bentuk utama, jadi regex perlu dua alternatif agar bisa menangkap keduanya.

**- [a-z]+ digunakan untuk menangkap named entity, yaitu entity berbasis nama.**

Contoh:

- &lt;
- &gt;
- &amp;

**- \#\d+ digunakan untuk menangkap numeric entity, yaitu entity berbasis angka.*

Contoh:

- &#128514;
- &#169;

Jadi, dua alternatif itu dipakai karena HTML entity bisa berupa nama atau angka. Kalau hanya memakai salah satu, sebagian entity tidak akan terdeteksi.

**c) Mengapa mention menggunakan lookbehind (?<![.\w@]) — apa yang diproteksi dari false positive?**

Jawab: Bagian ini mencegah false positive dari:

- email@domain.com
- @@username
- abc@user

Jadi @username hanya dianggap mention kalau sebelum @ bukan:

- titik (.)
- huruf/angka/underscore
- tanda @

**d) re.VERBOSE | re.IGNORECASE | re.MULTILINE — jelaskan efek MASING-MASING flag.**

Jawab:

**- re.VERBOSE**: Membuat regex bisa ditulis rapi dengan spasi, baris baru, dan komentar.

**- re.IGNORECASE**: Membuat regex tidak membedakan huruf besar dan kecil.

Contoh:

- GPT-4o
- gpt-4o

keduanya bisa match.

**- re.MULTILINE**: Membuat ^ dan $ bekerja per baris, bukan hanya awal dan akhir seluruh string.

Dalam soal ini berguna untuk pola seperti ^RT agar bisa mendeteksi awal baris.

**e) Prediksi isi dictionary entities: entity_type apa saja yang ada dan isinya apa?**

Jawab:

{
  'rt_prefix': ['RT @budi_nlp: '],
  'html_entity': ['&lt;', '&gt;', '&lt;', '&gt;'],
  'ai_model': ['GPT-4o', 'LLaMA-3-70B'],
  'percentage': ['94.7%', '89.3%'],
  'stat_sig': ['p&lt;0.05'],
  'url': ['https://t.co/xDataset123', 'bit.ly/IndoNLU24'],
  'mention': ['@sari_ml99', '@andi_ai'],
  'hashtag': ['#NLP', '#BahasaIndonesia', '#LLM']
}